In [ ]:
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import zipfile
from shapely.geometry import box

# Routing
from r5py import TravelTimeMatrixComputer, TransportMode
import datetime
from datetime import timedelta

# R5
import r5py
from r5py import TransportNetwork

In [ ]:
zip_path = "scratch/euro-dem-tif.zip"

In [ ]:
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    files = zip_ref.namelist()

for f in files:
    print(f)

In [ ]:
# Paths
import zipfile
import os

outer_zip_path = "scratch/euro-dem-tif.zip"
inner_zip_name = "euro-dem-tif/data/eurodem.zip"
temp_inner_zip = "scratch/eurodem_extracted.zip"

# Extract inner zip to /scratch (where there's space)
with zipfile.ZipFile(outer_zip_path, 'r') as outer_zip:
    with outer_zip.open(inner_zip_name) as inner_zip_file:
        with open(temp_inner_zip, 'wb') as f:
            f.write(inner_zip_file.read())

print("Extracted inner ZIP to:", temp_inner_zip)

In [ ]:
with zipfile.ZipFile(temp_inner_zip, 'r') as zip_ref:
    files = zip_ref.namelist()
    tif_files = [f for f in files if f.endswith('.tif')]

print("TIFF files found:", tif_files)


In [ ]:
import rasterio

# Path to the inner zip that contains the .tif
tif_zip_path = "scratch/eurodem_extracted.zip"
tif_inside_zip = "eurodem.tif"

# Construct the GDAL virtual file path
vsi_path = f"/vsizip/{tif_zip_path}/{tif_inside_zip}"

# Open with rasterio
with rasterio.open(vsi_path) as src:
    print("Raster opened!")
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)
    print("Width x Height:", src.width, "x", src.height)

In [ ]:
import geopandas as gpd

hex_path = "./data/Tampere_region_boundary.geojson"
hex_gdf = gpd.read_file(hex_path)

hex_union = hex_gdf.unary_union

In [ ]:
hex_union_gdf = gpd.GeoDataFrame(geometry=[hex_union], crs=hex_gdf.crs)

In [ ]:
hex_union_gdf.explore()

In [ ]:
import rasterio
from rasterio.mask import mask

dem_zip_path = "scratch/eurodem_extracted.zip"
dem_tif_name = "eurodem.tif"
vsi_path = f"/vsizip/{dem_zip_path}/{dem_tif_name}"

with rasterio.open(vsi_path) as src:
    # Reproject hex union to DEM CRS
    hex_union_gdf = hex_union_gdf.to_crs(src.crs)

    # Mask DEM with hex union geometry
    out_image, out_transform = mask(src, hex_union_gdf.geometry, crop=True)
    out_meta = src.meta.copy()

out_meta.update({
    "driver": "GTiff",
    "height": out_image.shape[1],
    "width": out_image.shape[2],
    "transform": out_transform
})

output_path = "scratch/cropped_dem_tampere.tif"

with rasterio.open(output_path, "w", **out_meta) as dest:
    dest.write(out_image)

print(f"Cropped DEM saved to: {output_path}")




In [ ]:
import rasterio

cropped_path = "scratch/cropped_dem_tampere.tif"

with rasterio.open(cropped_path) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Width x Height:", src.width, "x", src.height)
    print("Resolution:", src.res)
    print("Number of bands:", src.count)

In [ ]:
import matplotlib.pyplot as plt

with rasterio.open(cropped_path) as src:
    img = src.read(1)  # read first band
    
plt.imshow(img, cmap='terrain')
plt.colorbar(label='Elevation (m)')
plt.title("Cropped DEM")
plt.show()

In [ ]:
print(r5py.__version__)

In [ ]:
## Network Santiago
# Saving the path for .pbf and gtfs archives into new variables
osm_tampere = "./data/Tampere_region.osm.pbf"
#gtfs_hsk = "./data/GTFS_helsinki.zip"
dem_tampere = "scratch/cropped_dem_tampere.tif"

# Build de network
network = TransportNetwork(osm_tampere,elevation_model=dem_tampere)

In [ ]:
tampere_shape = gpd.read_file("./data/h3_centroids_Tampere_Filtered.csv")

In [ ]:
tampere_shape

In [ ]:
from shapely.geometry import Point

# assuming dataframe is called df
tampere_shape["geometry"] = tampere_shape.apply(lambda row: Point(row["lon"], row["lat"]), axis=1)

# convert to GeoDataFrame
tampere_shape = gpd.GeoDataFrame(tampere_shape, geometry="geometry", crs="EPSG:4326")

In [ ]:
tampere_shape

In [ ]:
tampere_centroids = tampere_shape[["id","geometry"]]

In [ ]:
# Creating two df for origins and destinations
tampere_hexagons_9_origins = tampere_centroids.copy()
tampere_hexagons_9_destinations = tampere_centroids.copy()

In [ ]:
# Just the first origins
origins_subset = tampere_hexagons_9_origins.iloc[:500]

In [ ]:
tampere_hexagons_9_origins

In [ ]:
tampere_hexagons_9_destinations

In [ ]:
## Travel time matrix
travel_time_matrix_computer = TravelTimeMatrixComputer(
    network,
    origins=tampere_hexagons_9_origins,
    destinations=tampere_hexagons_9_destinations,
    departure=datetime.datetime(2024,3,23,8,0), # Thursday
    max_time = timedelta(minutes=45),
    departure_time_window = timedelta(minutes=5), # Using a window of 30 min 
    transport_modes=[TransportMode.BICYCLE],
    
)
travel_time_matrix = travel_time_matrix_computer.compute_travel_times()

In [ ]:
travel_time_matrix

### Trying Detail Itineraries

In [ ]:
travel_time_matrix

In [ ]:
# 1. Drop NaN values
ttm_clean = travel_time_matrix.dropna()

In [ ]:
ttm_clean

In [ ]:
ttm_clean.to_csv("./data/travel_time_matrix_bicycle_tampere.csv")

In [ ]:
# 2. Create DataFrame for from_id
df_from = ttm_clean[['from_id']].reset_index(drop=True)

# 3. Create DataFrame for to_id
df_to = ttm_clean[['to_id']].reset_index(drop=True)

In [ ]:
# Take first 100 unique from_id
df_from_100 = df_from.head(100)

# Take first 100 unique to_id
df_to_100 = df_to.head(1000)

In [ ]:
# Rename column 'ID' to 'id'
df_from_100 = df_from_100.rename(columns={"from_id": "id"})
df_to_100 = df_to_100.rename(columns={"to_id": "id"})


In [ ]:
import h3
import geopandas as gpd
from shapely.geometry import Point

def h3_to_centroid_gdf(df, h3_col):
    """Convert a DataFrame with an H3 index column to a GeoDataFrame of centroids."""
    df = df.copy()
    df["geometry"] = df[h3_col].apply(
        lambda h: Point(h3.h3_to_geo(h)[1], h3.h3_to_geo(h)[0])  # lon, lat
    )
    return gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

# Convert both
gdf_to_100 = h3_to_centroid_gdf(df_to_100, "id")
gdf_from_100 = h3_to_centroid_gdf(df_from_100, "id")

In [ ]:
gdf_to_100 = gdf_to_100.drop_duplicates()

In [ ]:
gdf_from_100 = gdf_from_100.head(1)

In [ ]:
detailed_itineraries = r5py.DetailedItineraries(
    network,
    origins=gdf_from_100,
    destinations=gdf_to_100,
    departure=datetime.datetime(2024,3,23,8,0),
    max_time = timedelta(minutes=45),
    departure_time_window = timedelta(minutes=5),
    transport_modes=[TransportMode.BICYCLE],
    force_all_to_all = False,
    snap_to_network=True,
)

In [ ]:
detailed_itineraries